# 05 · Competency questions – Nordvia Retail (SAP FI)

**Goal.** Answer the five competency questions of the challenge with one SPARQL query each, run on the knowledge graph.

**Pieces**
- `queries/cq1 … cq5 .rq` – the queries. Each file starts with the definitions and assumptions it relies on.
- `kg/nordvia-kg.ttl` and `ontology/nordvia-finance.ttl` – data and ontology, loaded together.
- `queries/results/` – the answer to each question, as CSV.

**Rules of the challenge applied here**
- Key date for open items and debt age: 31 Dec 2025.
- Debt-age buckets: not yet due · 1–30 · 31–60 · 61–90 · over 90 days overdue. They are individuals of the ontology; the query reads their limits from there.
- Euros: there is no rate table, so amounts are converted with the monthly rates derived in `03_gold` (section 2), stored in the graph with their method.

**How the queries read.** They use business terms only (customer, payee, cost center, due date). No SAP table or field name appears in them.

## 0 · Environment

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [2]:
import os, sys, time
from importlib.metadata import version
import pandas as pd
import pyoxigraph as ox          # SPARQL engine; installed together with morph-kgc

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Python", sys.version.split()[0], "|", ", ".join(f"{p} {version(p)}" for p in ["pyoxigraph", "pandas"]))

Python 3.12.7 | pyoxigraph 0.3.22, pandas 2.3.3


## 1 · Load the graph
Data and ontology go into the same store, so a query can use what the ontology says: class hierarchy, labels, the debt-age buckets, the net days of each payment term.

In [3]:
store = ox.Store()
for path in ["kg/nordvia-kg.ttl", "ontology/nordvia-finance.ttl"]:
    with open(path, "rb") as f:
        store.load(f, "text/turtle")
print(f"{len(store):,} triples loaded")

def to_python(term):
    """RDF term -> Python value (numbers as numbers, everything else as text)."""
    if term is None:
        return None
    if isinstance(term, ox.Literal):
        kind = term.datatype.value.rsplit("#", 1)[-1]
        if kind in ("integer", "int", "long"):
            return int(term.value)
        if kind in ("decimal", "double", "float"):
            return float(term.value)
    return term.value

def ask(name, show=True):
    """Run queries/<name>.rq and return the answer as a table."""
    text = open(f"queries/{name}.rq", encoding="utf-8").read()
    if show:
        print(text)
    start = time.time()
    result = store.query(text)
    columns = [v.value for v in result.variables]
    table = pd.DataFrame([[to_python(row[c]) for c in columns] for row in result], columns=columns)
    print(f"-- {len(table)} rows in {time.time() - start:.2f} s")
    os.makedirs("queries/results", exist_ok=True)
    table.to_csv(f"queries/results/{name}.csv", index=False, encoding="utf-8")
    return table

120,443 triples loaded


## 2 · CQ1 – Outstanding receivables per customer, by debt age
*What is the outstanding receivables balance per customer, by debt age?*

In [4]:
cq1 = ask("cq1_receivables_by_debt_age")

# CQ1 - What is the outstanding receivables balance per customer, by debt age?
#
# Open item   : a customer line not cleared on or before the key date.
# Debt age    : days from the due date of the item to the key date. The limits of the
#               buckets are read from the ontology (nvf:DebtAgeBucket), not written here.
# Credit memos: open credits reduce what the customer owes but have no meaningful age,
#               so they are reported in their own column instead of an age bucket.
# Amounts     : signed (debit +, credit -), in the local currency of the company and in euros.
# Reversed invoices need no filter: each one is cleared by its reversal, so it is not open.

PREFIX nvf:     <https://nordvia.com/ontology/finance#>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:     <http://www.w3.org/2001/XMLSchema#>
PREFIX skos:    <http://www.w3.org/2004/02/skos/core#>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX foaf:    <http://xmlns.com/foaf/0.1/>

SELEC

**Totals per bucket.** Amounts in euros, so the three companies can be added. Open credits are credit memos not yet applied: they reduce the balance and are kept out of the age buckets.

In [5]:
order = cq1.drop_duplicates("bucket").sort_values("bucketOrder")["bucket"].tolist()
by_bucket = cq1.groupby("bucket").agg(items=("items", "sum"), customers=("customerNumber", "nunique"), amount_eur=("amountEur", "sum")).loc[order]
by_bucket.loc["Net receivables"] = [by_bucket["items"].sum(), cq1["customerNumber"].nunique(), by_bucket["amount_eur"].sum()]
by_bucket.astype({"items": int, "customers": int})

,items,customers,amount_eur
bucket,,,
Not yet due,44,30,"378,332.76"
1-30 days overdue,18,17,"150,139.23"
31-60 days overdue,10,8,"104,024.20"
61-90 days overdue,2,2,"19,896.89"
Over 90 days overdue,15,8,"98,371.70"
Open credits,22,17,"-22,484.14"
Net receivables,111,61,"728,280.64"


**Per customer.** One row per customer with an open balance, in the local currency of its company; the last column is the balance in euros. One-time and unregistered customers are identified by their kind, since they are not ordinary customers.

In [6]:
per_customer = cq1.pivot_table(index=["company", "currency", "customerNumber", "customerName", "customerKind"], columns="bucket",
                               values="amount", aggfunc="sum", fill_value=0, dropna=False, observed=True)
per_customer = per_customer[[b for b in order if b in per_customer.columns]]
per_customer = per_customer[(per_customer != 0).any(axis=1)]
per_customer["Balance"] = per_customer.sum(axis=1)
per_customer["Balance (EUR)"] = cq1.groupby(["company", "currency", "customerNumber", "customerName", "customerKind"], dropna=False)["amountEur"].sum()
per_customer.to_csv("queries/results/cq1_receivables_per_customer.csv", encoding="utf-8")
per_customer.reset_index().fillna({"customerName": "(no master record)"}).set_index(["company", "customerNumber"])

bucket                 currency                    customerName           customerKind  Not yet due  1-30 days overdue  31-60 days overdue  61-90 days overdue  Over 90 days overdue  Open credits   Balance  \
company customerNumber                                                                                                                                                                                         
1000    0000100004          EUR    Meridian Distribuciones S.L.      Domestic Customer     7,879.87               0.00                0.00                0.00                  0.00          0.00  7,879.87   
        0000100008          EUR              Maple Trading S.A.      Domestic Customer     4,872.67               0.00                0.00                0.00                  0.00          0.00  4,872.67   
        0000100009          EUR      Nova Distribuciones S.L.U.      Domestic Customer    24,968.91               0.00                0.00                0.00                  0.00          0.00 24,968.91   
        0000100014          EUR                  Nexo Home S.A.      Domestic Customer    14,614.20               0.00           44,383.92                0.00                  0.00          0.00 58,998.12   
        0000100015          EUR            Alpha Trading S.L.U.      Domestic Customer    16,964.27               0.00                0.00                0.00                  0.00          0.00 16,964.27   
        0000100019          EUR              Aurora Market S.L.      Domestic Customer         0.00          13,196.01                0.00                0.00                  0.00          0.00 13,196.01   
        0000100024          EUR       Crest Distribuciones S.L.      Domestic Customer     4,978.10           4,769.01                0.00                0.00                  0.00          0.00  9,747.11   
        0000100025          EUR             Granite Brands S.L.      Domestic Customer         0.00               0.00                0.00                0.00                  0.00     -2,392.06 -2,392.06   
        0000100030          EUR       Prime Distribuciones S.A.      Domestic Customer         0.00          10,122.35                0.00                0.00                  0.00          0.00 10,122.35   
        0000100031          EUR                  Iber Home S.A.      Domestic Customer         0.00               0.00            2,880.24                0.00                  0.00          0.00  2,880.24   
        0000100035          EUR             Silver Trading S.A.      Domestic Customer         0.00           3,114.42                0.00                0.00              6,958.99          0.00 10,073.41   
        0000100039          EUR            Oak Comercial S.L.U.      Domestic Customer     2,197.44               0.00                0.00                0.00                  0.00          0.00  2,197.44   
        0000100040          EUR               Prime Market S.A.      Domestic Customer         0.00           5,728.50                0.00                0.00                  0.00          0.00  5,728.50   
        0000100042          EUR              Prime Trading S.A.      Domestic Customer    53,526.21               0.00                0.00                0.00                  0.00     -1,285.37 52,240.84   
        0000100044          EUR             Nexo Trading S.L.U.      Domestic Customer    12,316.21          12,811.06                0.00                0.00                  0.00       -561.88 24,565.39   
        0000100046          EUR            Bright Outlet S.L.U.      Domestic Customer         0.00               0.00                0.00                0.00              9,498.97          0.00  9,498.97   
        0000100047          EUR             Aurora Trading S.L.      Domestic Customer    19,001.59               0.00                0.00                0.00                  0.00          0.00 19,001.59   
        0000100048          EUR         Meridian

## 3 · CQ2 – Vendors that invoiced the most in 2025
*Which vendors invoiced us the most in 2025, by company code and in euros?*

In [7]:
cq2 = ask("cq2_vendor_invoicing")

# CQ2 - Which vendors invoiced us the most in 2025, by company code and in euros?
#
# Vendor   : a payee of class nvf:Vendor. Employees reimbursed through the payables
#            ledger are payees too, but not vendors, so they are left out.
# Invoiced : the amount credited to the vendor on payable invoices of fiscal year 2025,
#            that is, the gross amount of the invoice including tax.
# Reversed invoices are left out; their reversals are not invoices and never enter.
# Euros    : nvf:amountInGroupCurrency, converted with the monthly rate of the posting period.
# The two records of the same vendor (same tax ID) are one payee, so its invoices add up.

PREFIX nvf:     <https://nordvia.com/ontology/finance#>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX foaf:    <http://xmlns.com/foaf/0.1/>

SELECT ?company ?vendorNumber ?vendorName (COUNT(DISTINCT ?invoice) AS ?invoices) (SUM(?euros) AS ?invoicedEur)
WHERE {
  ?invoice a nvf:PayableInvoice ;
           nvf:fiscalYear 2025

**Top five per company code**, with the share each vendor has of what its company was invoiced.

In [8]:
cq2["share_of_company"] = cq2["invoicedEur"] / cq2.groupby("company")["invoicedEur"].transform("sum") * 100
cq2["rank"] = cq2.groupby("company")["invoicedEur"].rank(ascending=False).astype(int)
display(cq2[cq2["rank"] <= 5].set_index(["company", "rank"]))
cq2.groupby("company").agg(vendors=("vendorNumber", "size"), invoices=("invoices", "sum"), invoiced_eur=("invoicedEur", "sum"))

vendorNumber              vendorName  invoices  invoicedEur  share_of_company
company rank                                                                              
1000    1      0000200005      Zenith Energy S.L.        13   191,487.61             11.16
        2      0000200006     Maple Supplies S.L.         9   168,046.42              9.79
        3      0000200009         Nova Foods S.L.         9   152,096.80              8.86
        4      0000200022         Iber Media S.A.         7   148,423.98              8.65
        5      0000200003       Zenith Foods S.A.         9   136,701.44              7.97
2000    1      0000200033       Summit Foods Inc.         7   102,456.38             14.37
        2      0000200030   Cobalt Textiles Corp.         9    93,646.29             13.13
        3      0000200037        Falcon Foods LLC        10    92,114.45             12.92
        4      0000200031     Alpha Logistics LLC         8    86,250.27             12.10
        5      0000200029  Northstar Office Corp.         7    81,100.63             11.37
3000    1      0000200051       Maple Energy Ltd.         6   151,491.94             17.45
        2      0000200050        Orion Media Ltd.         7   146,229.45             16.85
        3      0000200053    Cobalt Supplies Ltd.         8   145,527.54             16.77
        4      0000200055  Zenith Consulting Ltd.         6   129,965.57             14.97
        5      0000200056  Prime Electronics Ltd.         3    74,491.68              8.58

,vendors,invoices,invoiced_eur
company,,,
1000,28,215,"1,716,112.40"
2000,21,140,"713,092.88"
3000,16,87,"867,981.03"


## 4 · CQ3 – Cost absorbed by each cost center, by type of expense
*How much cost has each cost center absorbed, by type of expense?*

In [9]:
cq3 = ask("cq3_cost_by_cost_center")

# CQ3 - How much cost has each cost center absorbed, by type of expense?
#
# Cost            : lines posted to an account on which the company requires a cost center
#                   (nvf:CostCenterRequiredAccount). Debits add cost, credits take it away.
# Type of expense : the G/L account of the line.
# No cost center  : a few lines have none although the account requires it; they are kept
#                   and shown under "(no cost center)" so the total stays complete.
# Reversal documents and the documents they reverse are left out.
# Amounts in the local currency of the company and in euros.

PREFIX nvf:     <https://nordvia.com/ontology/finance#>
PREFIX skos:    <http://www.w3.org/2004/02/skos/core#>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?company ?costCenter ?costCenterName ?account ?expenseType ?currency
       (SUM(?signed) AS ?cost) (SUM(?signedEur) AS ?costEur) (COUNT(?line) AS ?lines)
WHERE {
  ?line nvf:lineItemOf ?document ;
        nvf:glAccount ?glAccou

**Cost centers against types of expense**, in euros. The rows "(no cost center)" are the postings that should have had a cost center and do not.

In [10]:
matrix = cq3.pivot_table(index=["company", "costCenter", "costCenterName"], columns="expenseType", values="costEur", aggfunc="sum", fill_value=0)
matrix["Total"] = matrix.sum(axis=1)
matrix.to_csv("queries/results/cq3_cost_center_by_expense_type.csv", encoding="utf-8")
display(matrix)
cq3.groupby(["company", "currency"]).agg(cost_local=("cost", "sum"), cost_eur=("costEur", "sum"), lines=("lines", "sum"))

expenseType                                        Bank fees  Consulting fees  Depreciation expense  IT services  Marketing  Office supplies      Rent  Salaries and wages  Social security  Travel expenses  Utilities  \
company costCenter       costCenterName                                                                                                                                                                                   
1000    (no cost center) (no cost center)               0.00           229.21                  0.00         0.00       0.00             0.00      0.00                0.00             0.00             0.00       0.00   
        0000011010       Administration                 0.00         6,909.92             14,760.00         0.00       0.00         3,583.70 17,355.22           75,128.68        22,538.59         1,914.32       0.00   
        0000011020       Finance                    3,672.17        11,492.17                  0.00         0.00       0.00         3,424.94      0.00           63,015.58        18,904.68             0.00       0.00   
        0000011030       IT                             0.00             0.00             14,760.00    15,825.56       0.00             0.00      0.00           75,134.14        22,540.24             0.00       0.00   
        0000011040       Marketing                      0.00             0.00                  0.00         0.00  24,674.20             0.00      0.00           49,820.43        14,946.13         1,657.58       0.00   
        0000011050       Sales                          0.00             0.00                  0.00         0.00       0.00             0.00      0.00           93,461.03        28,038.32         5,392.85       0.00   
        0000011060       Logistics                      0.00             0.00                  0.00         0.00       0.00             0.00      0.00           81,470.53        24,441.16             0.00   4,829.03   
        0000011070       Store Barcelona Diagonal       0.00             0.00              9,840.00         0.00       0.00             0.00 23,665.01           99,241.37        29,772.42             0.00   2,333.72   
        0000011080       Store Valencia Centro          0.00             0.00              5,740.00         0.00       0.00             0.00  4,311.61           50,923.86        15,277.16             0.00     469.71   
        0000011090       Store Valencia Ruzafa          0.00             0.00              4,100.00         0.00       0.00             0.00 21,108.24           36,763.04        11,028.91             0.00   1,073.96   
2000    (no cost center) (no cost center)               0.00             0.00                  0.00         0.00     141.26             0.00  3,911.39                0.00             0.00             0.00       0.00   
        0000021010       Administration                 0.00         8,231.70             17,030.47         0.00       0.00         2,676.85 17,270.25           88,897.13         6,800.65         1,600.26       0.00   
        0000021020       Finance                    2,586.77         5,109.41                  0.00         0.00       0.00         9,227.52      0.00           74,338.64         5,686.89             0.00       0.00   
        0000021030       IT                             0.00             0.00             17,030.47     6,991.83       0.00             0.00      0.00           88,735.15         6,788.24             0.00       0.00   
        0000021040       Marketing                      0.00             0.00                  0.00         0.00  14,045.78             0.00      0.00           59,290.18         4,535.72           490.69       0.00   
        0000021050       Sales                          0.00             0.00                  0.00         0.00       0.00             0.00      0.00          111,246.20         8,510.34         2,980.10       0.00   
        0000021060       Logistics                      0.

,,cost_local,cost_eur,lines
company,currency,,,
1000,EUR,"1,015,569.39","1,015,569.39",386
2000,USD,"1,071,606.40","974,252.69",341
3000,GBP,"546,391.79","643,595.97",313


## 5 · CQ4 – Days customers take to pay, against their terms
*On average, how long do customers take to pay compared with their payment terms?*

In [11]:
cq4 = ask("cq4_days_to_pay")

# CQ4 - On average, how long do customers take to pay compared with their payment terms?
#
# Invoices     : customer invoice lines cleared by a customer payment. An invoice cleared by
#                its own reversal was cancelled, not paid, and is left out by that condition.
# Days to pay  : from the baseline date of the invoice to the clearing date (the payment).
# Terms        : the net days of the payment terms on the invoice line, which always match
#                the terms agreed with the customer in its master record.
# Days late    : days to pay minus net days; negative means paid before it was due.

PREFIX nvf:  <https://nordvia.com/ontology/finance#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?paymentTerms ?termsName ?netDays
       (COUNT(?line) AS ?invoicesPaid)
       (AVG(?daysToPay) AS ?avgDaysToPay)
       (AVG(?daysToPay - ?netDays) AS ?avgDaysLate)
       (SUM(IF(?daysToPay > ?netDays, 1, 0)) AS 

In [12]:
cq4["share_paid_late"] = cq4["paidLate"] / cq4["invoicesPaid"] * 100
n = cq4["invoicesPaid"]
overall = pd.DataFrame([{"paymentTerms": "All", "termsName": "All paid invoices", "netDays": (cq4["netDays"] * n).sum() / n.sum(),
                         "invoicesPaid": n.sum(), "avgDaysToPay": (cq4["avgDaysToPay"] * n).sum() / n.sum(),
                         "avgDaysLate": (cq4["avgDaysLate"] * n).sum() / n.sum(), "paidLate": cq4["paidLate"].sum(),
                         "share_paid_late": cq4["paidLate"].sum() / n.sum() * 100}])
pd.concat([cq4, overall], ignore_index=True).set_index("paymentTerms")

,termsName,netDays,invoicesPaid,avgDaysToPay,avgDaysLate,paidLate,share_paid_late
paymentTerms,,,,,,,
0001,Payable immediately,0.00,215,7.61,7.61,130,60.47
NT15,Net 15 days,15.00,58,33.62,18.62,41,70.69
NT30,Net 30 days,30.00,204,37.40,7.40,128,62.75
NT60,Net 60 days,60.00,113,67.77,7.77,71,62.83
All,All paid invoices,23.34,590,31.99,8.65,370,62.71


`avgDaysToPay` is the time customers take; `netDays` is what their terms allow; `avgDaysLate` is the difference. In the row *All*, `netDays` is the average term of the invoices paid.

## 6 · CQ5 – Profit and loss per company code and month
*What is the profit and loss (revenue – expenses) per company code and month?*

In [13]:
cq5 = ask("cq5_profit_and_loss")

# CQ5 - What is the profit and loss (revenue - expenses) per company code and month?
#
# Scope    : lines posted to profit and loss accounts. Credits add to the result,
#            debits take away from it.
# Revenue  : the source has no field that separates revenue accounts from expense accounts.
#            An account counts as revenue when its balance over the whole year is a credit
#            (sales and exchange-rate gains), and as expense otherwise.
# Month    : the fiscal period of the document, which equals the month of its posting date.
# Reversal documents and the documents they reverse are left out, so an amount posted in
# one month and cancelled in the next appears in neither.
# Amounts in the local currency of the company; the result also in euros.

PREFIX nvf:     <https://nordvia.com/ontology/finance#>
PREFIX skos:    <http://www.w3.org/2004/02/skos/core#>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?company ?month ?currency
       (SUM(IF(?isRevenue, ?signed

**Result per month**, each company in its local currency, and the group in euros.

In [14]:
monthly = cq5.pivot(index="month", columns="company", values="result")
monthly.columns = [f"{c} ({cq5.loc[cq5['company'] == c, 'currency'].iloc[0]})" for c in monthly.columns]
monthly["Group (EUR)"] = cq5.groupby("month")["resultEur"].sum()
monthly.loc["Year"] = monthly.sum()
display(monthly)
cq5.groupby(["company", "currency"])[["revenue", "expenses", "result", "resultEur"]].sum()

,1000 (EUR),2000 (USD),3000 (GBP),Group (EUR)
month,,,,
1,"35,315.53","36,036.78","30,924.54","107,010.43"
2,"-49,990.27","110,420.47","108,149.83","184,547.92"
3,-357.10,"10,238.96","-15,907.28","-9,507.05"
4,"61,688.90","88,967.72","-27,105.34","112,854.18"
5,"34,075.75","-48,209.16",-500.08,"-11,073.39"
6,"-62,233.75","-6,261.11","-65,671.89","-145,207.55"
7,"88,649.50","39,661.58","-36,809.27","81,273.84"
8,"80,383.59","-69,354.54","-93,140.29","-90,696.98"
9,"54,148.50","-17,219.93","45,623.52","92,450.63"


,,revenue,expenses,result,resultEur
company,currency,,,,
1000,EUR,"2,794,507.59","2,300,553.49","493,954.10","493,954.10"
2000,USD,"1,932,031.15","1,737,474.08","194,557.07","184,510.39"
3000,GBP,"1,079,218.57","1,075,905.44","3,313.13","4,974.27"


## 7 · Assumptions behind the answers
The challenge asks to state the assumptions made where the data is ambiguous. Each query file repeats the ones it uses.

| Topic | Assumption |
|---|---|
| Reference date | The posting date decides the year and the key date; the due date (baseline date plus net days) decides the debt age |
| Open item | A customer line with no clearing on or before the key date |
| Open credit memos | Reduce the customer's balance; shown in their own column because a credit has no meaningful age |
| Reversals | A reversal and the document it cancels are left out of invoicing, cost and profit and loss |
| Opening balances | Hold only bank, fixed assets and equity, so they touch none of the five answers |
| Vendors | Employees reimbursed through the payables ledger are not vendors; the two records of the same vendor count as one |
| Amount invoiced | The gross amount credited to the vendor, tax included |
| Probable duplicate invoice | Counted, since it was posted and paid twice; whether to exclude it is a business decision |
| Cost | Postings to accounts that require a cost center; the few without one are kept and shown apart |
| Expired cost center | Its postings after the end date are counted where they were posted |
| Payment terms | Those on the invoice line, which always match the customer master |
| Revenue and expense | The source does not separate the accounts: an account with a credit balance over the year is revenue |
| Euros | Monthly rate of the posting period, derived from the documents (direct, inverse or cross rate) |